# Lab 3B: Agentic RAG (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 3 | about 10 minutes

- Give a model two tools: document search and a safe calculator
- Let a planner choose the next step, one tool call at a time
- Run the loop with guardrails and read the Thought / Action / Observation trace

## 1. Tools

In [ ]:
import os, re, glob, json, ast, operator, warnings
import numpy as np
from fastembed import TextEmbedding
from llm_client import call_llm, llm_available, llm_mode
warnings.filterwarnings("ignore")

DOCS = []
for f in sorted(glob.glob("data/corpus/*.txt")):
    head, body = open(f, encoding="utf-8").read().split("\n\n", 1)
    meta = dict(l.split(": ", 1) for l in head.splitlines() if ": " in l)
    DOCS.append(dict(title=meta["title"], text=body.strip()))
CHUNKS = []
for d in DOCS:
    w = re.findall(r"\S+\s*", d["text"])
    CHUNKS += [dict(title=d["title"], text="".join(w[i:i + 150]).strip()) for i in range(0, len(w), 150)]
_model = TextEmbedding("BAAI/bge-small-en-v1.5")
def embed(texts):
    v = np.array(list(_model.embed(list(texts))), dtype=float)
    return v / np.linalg.norm(v, axis=1, keepdims=True)
CVEC = embed([c["text"] for c in CHUNKS])

def search_docs(query, k=3):
    """Top-k 150-word chunks by cosine similarity, each with its source title and score."""
    sims = CVEC @ embed([query])[0]
    top = np.argsort(-sims)[:k]
    return "\n".join(f"[{CHUNKS[i]['title']}] (sim {sims[i]:.2f}) {CHUNKS[i]['text']}" for i in top)

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}
def calculator(expression):
    """Safe arithmetic only: numbers and + - * / ** % (percent signs are not supported)."""
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only plain arithmetic is allowed")
    return str(round(ev(ast.parse(expression, mode="eval").body), 4))

TOOLS = {"search_docs": (search_docs, "search the company documents; input = a search query"),
         "calculator": (calculator, "evaluate arithmetic such as 210 * 3; input = an expression")}
print(calculator("0.15 * 2400"), "|", [n for n in TOOLS])

## 2. Planner
Free LLM? Set `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL` (same as Day 2) in the commented cell below, run it, then run the planner cell.
No key: an OFFLINE rule-based stand-in plans instead (it is not a language model).

In [ ]:
# os.environ["LLM_API_KEY"]  = "PASTE-YOUR-KEY"
# os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # Google AI Studio: https://generativelanguage.googleapis.com/v1beta/openai/   Ollama: http://localhost:11434/v1
# os.environ["LLM_MODEL"]    = "PASTE-A-MODEL-NAME"

In [ ]:
NUM = r"\d[\d,]*\.?\d*"
def _num(s): return float(s.replace(",", ""))

def plan_offline(question, scratch):
    """OFFLINE rule-based stand-in for the LLM planner."""
    searched = [s for s in scratch if s["action"] == "search_docs"]
    calcs = [s for s in scratch if s["action"] == "calculator"]
    q = question.lower()
    pure_math = bool(re.search(NUM, q)) and not re.search(r"[a-z]{4,}", re.sub(r"what|is|of|how|much|percent|times|plus|minus|double|triple|nights?|days?|weeks?", "", q))
    has_arith = bool(re.search(r"\b(double|triple|twice|times|total|cost|nights?|days?|percent)\b|%|\d\s*[-+*/x]\s*\d", q))
    if calcs: return dict(thought="I have the calculation result.", action="finish", input="")
    if pure_math:
        m = re.search(rf"({NUM})\s*(?:%|percent)\s*of\s*({NUM})", q)
        expr = f"{_num(m.group(1)):g} / 100 * {_num(m.group(2)):g}" if m else re.sub(r"[^0-9+\-*/(). ]", "", q)
        return dict(thought="The question is pure arithmetic, no documents needed.", action="calculator", input=expr)
    if not searched:
        return dict(thought="I need facts from the documents first.", action="search_docs", input=question)
    obs = searched[-1]["observation"]
    if has_arith and (best := re.search(r"sim (0\.\d+)", obs)) and float(best.group(1)) >= 0.60:
        qw = set(re.findall(r"[a-z]{4,}", q))
        top = re.split(r"\n(?=\[[^\]]+\] \(sim)", obs)[0]  # best-ranked chunk only
        lines = [re.sub(r"^\[.*?\] \(sim [\d.]+\)", "", l) for l in top.splitlines()]
        lines = [l for l in lines if re.search(NUM, l)]
        line = max(lines, key=lambda l: len(qw & set(re.findall(r"[a-z]{4,}", l.lower()))))
        fact = _num(re.search(NUM, line).group())
        word = next((w for w in ("double", "triple", "twice") if w in q), None)
        n = re.search(rf"({NUM})\s*(?:nights?|days?)", q)
        mult = {"double": 2, "twice": 2, "triple": 3}.get(word) or (_num(n.group(1)) if n else 1)
        return dict(thought=f"The documents give {fact:g}; now multiply by {mult:g}.", action="calculator", input=f"{fact:g} * {mult:g}")
    return dict(thought="Nothing useful or numeric found to compute from.", action="finish", input="")

def plan_llm(question, scratch):
    tools = "\n".join(f"- {n}: {d}" for n, (_, d) in TOOLS.items()) + "\n- finish: give the final answer"
    pad = "\n".join(f"Step {i}: {s['action']}({s['input']}) -> {s['observation'][:300]}" for i, s in enumerate(scratch, 1)) or "(empty)"
    prompt = (f"Tools:\n{tools}\n\nScratchpad:\n{pad}\n\nQUESTION: {question}\n"
              'Reply with JSON only: {"thought": "...", "action": "<tool name or finish>", "input": "..."}')
    try:
        out = json.loads(call_llm(prompt, system="You are a careful planner. Pick ONE next action.", json_mode=True))
        if out.get("action") in list(TOOLS) + ["finish"]: return dict(thought=out.get("thought", ""), action=out["action"], input=str(out.get("input", "")))
        return dict(thought=f"Planner chose a disallowed action: {out.get('action')!r}", action="finish", input="")
    except Exception as e:
        return dict(thought=f"Planner output could not be parsed ({type(e).__name__})", action="finish", input="")

def plan(question, scratchpad):
    return plan_llm(question, scratchpad) if llm_available() else plan_offline(question, scratchpad)

print("LLM:", llm_mode())
print("=== PLANNER IN USE:", "LLM planner (JSON mode)" if llm_available() else "OFFLINE rule-based stand-in (not a language model)", "===")

## 3. Agent loop with guardrails
Guardrails: at most 4 steps, only allow-listed tools, read-only tools (no side effects), every step printed.

In [ ]:
MAX_STEPS = 4
def run_agent(question):
    scratch = []
    print(f"\nTASK: {question}")
    for step in range(1, MAX_STEPS + 1):
        p = plan(question, scratch)
        print(f"\n[{step}] Thought: {p['thought']}\n    Action: {p['action']}({p['input']})")
        if p["action"] == "finish": break
        if p["action"] not in TOOLS: print("    blocked: tool not on allow-list"); break
        try: obs = TOOLS[p["action"]][0](p["input"])
        except Exception as e: obs = f"ERROR: {e}"
        print("    Observation:", obs[:160].replace("\n", " ") + ("..." if len(obs) > 160 else ""))
        scratch.append(dict(p, observation=obs))
    return final_answer(question, scratch)

def final_answer(question, scratch):
    calc = next((s for s in reversed(scratch) if s["action"] == "calculator"), None)
    search = next((s for s in scratch if s["action"] == "search_docs"), None)
    if llm_available():
        pad = "\n".join(f"{s['action']}({s['input']}) -> {s['observation'][:600]}" for s in scratch)
        return call_llm(f"{pad}\n\nQUESTION: {question}", system="Answer only from the scratchpad; if it lacks the answer say 'I could not find this'.")
    if search and not calc and not re.search(r"sim 0\.[6-9]", search["observation"]): return "I could not find this in the provided documents."
    if calc and search: return f"Using the figure from the documents ({calc['input'].split(' * ')[0]}), the result is {calc['observation']}."
    if calc: return f"The result is {calc['observation']}."
    return "I could not find this in the provided documents."

## 4. Three tasks

In [ ]:
for task in ["What is the hotel cap per night in London, and what would 3 nights cost?",
             "What is 15% of 2400?",
             "Who won the 1998 world cup?"]:
    print("FINAL ANSWER:", run_agent(task))

**Recap**
- An agent = LLM + tools + memory (the scratchpad) + a loop + guardrails.
- The guardrails here: 4-step cap, allow-listed read-only tools, safe calculator, visible trace.
- Offline, the planner is a rule-based stand-in; with a key, the LLM chooses each step.
- Day 4 adds MCP (a standard way to plug in tools) and agent design patterns.